## ⚠️ CELL 0 — Environment Check & Self-Repair
Checks if PyTorch is importable. If corrupted (e.g. after a broken pip reinstall), auto-repairs.


In [ ]:
import sys, subprocess
try:
    import torch
    print(f'[INFO] Python  : {sys.version.split()[0]}')
    print(f'[INFO] PyTorch : {torch.__version__}')
    print(f'[INFO] CUDA    : {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'[INFO] GPU     : {torch.cuda.get_device_name(0)}')
        print(f'[INFO] VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
except Exception as e:
    print(f'[ERROR] {e}')
    print('[FIX] Reinstalling PyTorch (default Kaggle version)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
        '--force-reinstall', 'torch', 'torchvision'])
    print('\n*** RESTART KERNEL NOW, THEN RUN ALL CELLS ***')

# 🚀 V4.0 — RoadFocusNet: FocusFormer + FocusMIM
| Module | Detail |
|--------|--------|
| FocusFormer Encoder | Windowed Focused Self-Attention (W-FSA, win=8) + Channel Self-Attention (CSA) |
| FPN Decoder | Multi-scale lateral fusion → 512×512 binary mask |
| Phase 1 FocusMIM | Self-supervised masked road-patch reconstruction |
| Phase 2 Fine-Tune | 50 epochs, AMP, cosine LR, gradient clipping |


## ⚙️ 1. Imports & Setup


In [ ]:
import os, gc, time, random, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2

print(f'[OK] PyTorch: {torch.__version__}  |  Albumentations: {A.__version__}')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = (device.type == 'cuda')
print(f'Device: {device}  |  AMP: {USE_AMP}')
if device.type == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

try:
    def autocast():    return torch.amp.autocast('cuda', enabled=USE_AMP)
    def make_scaler(): return torch.amp.GradScaler('cuda', enabled=USE_AMP)
except AttributeError:
    def autocast():    return torch.cuda.amp.autocast(enabled=USE_AMP)
    def make_scaler(): return torch.cuda.amp.GradScaler(enabled=USE_AMP)

print('[OK] Setup complete.')

## 🏗️ 2. FocusFormer Architecture
**Why windowed attention?** Stage-1 tokens = 128×128 = 16,384.
Full attention = 16K×16K → **~6 GB VRAM**. Window size 8 → 64-token windows → **~150 MB**.


In [ ]:
# ── Window helpers ────────────────────────────────────────────────────────
def window_partition(x, ws):
    B, H, W, C = x.shape
    x = x.view(B, H//ws, ws, W//ws, ws, C)
    return x.permute(0,1,3,2,4,5).contiguous().view(-1, ws*ws, C)

def window_reverse(windows, ws, H, W):
    nW = (H//ws) * (W//ws)
    B  = windows.shape[0] // nW
    x  = windows.view(B, H//ws, W//ws, ws, ws, -1)
    return x.permute(0,1,3,2,4,5).contiguous().view(B, H, W, -1)

# ── Channel Self-Attention (CSA) ──────────────────────────────────────────
class CSA(nn.Module):
    def __init__(self, dim, r=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(dim, dim//r, bias=False), nn.GELU(),
            nn.Linear(dim//r, dim, bias=False), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x.mean(1, keepdim=True))

# ── Windowed Focused Self-Attention (W-FSA) ───────────────────────────────
class WFSA(nn.Module):
    def __init__(self, dim, ws=8, heads=4, drop=0.):
        super().__init__()
        assert dim % heads == 0
        self.ws, self.heads = ws, heads
        self.hd    = dim // heads
        self.scale = self.hd ** -0.5
        self.qkv   = nn.Linear(dim, 3*dim, bias=True)
        self.gate  = nn.Sequential(
            nn.Linear(dim, dim//2), nn.GELU(),
            nn.Linear(dim//2, heads), nn.Sigmoid())
        self.proj  = nn.Linear(dim, dim)
        self.drop  = nn.Dropout(drop)

    def forward(self, x, H, W):
        B, _, C = x.shape
        sp = x.view(B, H, W, C)
        ph = (-H) % self.ws;  pw = (-W) % self.ws
        if ph or pw:
            sp = F.pad(sp, (0,0, 0,pw, 0,ph))
        Hp, Wp = sp.shape[1], sp.shape[2]
        wins = window_partition(sp, self.ws)
        Bw, N, _ = wins.shape
        qkv = self.qkv(wins).reshape(Bw, N, 3, self.heads, self.hd).permute(2,0,3,1,4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2,-1)) * self.scale
        attn = attn.softmax(-1)
        attn = self.drop(attn)
        g = self.gate(wins).permute(0,2,1).unsqueeze(-1)
        attn = attn * g
        out = (attn @ v).transpose(1,2).reshape(Bw, N, C)
        out = self.drop(self.proj(out))
        out = window_reverse(out, self.ws, Hp, Wp)
        if ph or pw:
            out = out[:, :H, :W, :].contiguous()
        return out.view(B, H*W, C)

# ── FocusFormer Block (CSA → W-FSA → MLP) ────────────────────────────────
class FocusFormerBlock(nn.Module):
    def __init__(self, dim, heads, ws=8, mlp_r=4., drop=0.):
        super().__init__()
        self.n1   = nn.LayerNorm(dim)
        self.csa  = CSA(dim)
        self.n2   = nn.LayerNorm(dim)
        self.wfsa = WFSA(dim, ws, heads, drop)
        self.n3   = nn.LayerNorm(dim)
        hid = int(dim * mlp_r)
        self.mlp  = nn.Sequential(
            nn.Linear(dim, hid), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hid, dim), nn.Dropout(drop))
    def forward(self, x, H, W):
        x = x + self.csa(self.n1(x))
        x = x + self.wfsa(self.n2(x), H, W)
        x = x + self.mlp(self.n3(x))
        return x

# ── Patch Embed & Merging ─────────────────────────────────────────────────
class PatchEmbed(nn.Module):
    def __init__(self, in_c=3, dim=96, ps=4):
        super().__init__()
        self.proj = nn.Conv2d(in_c, dim, ps, ps)
        self.norm = nn.LayerNorm(dim)
    def forward(self, x):
        x = self.proj(x)
        B, C, H, W = x.shape
        return self.norm(x.flatten(2).transpose(1,2)), H, W

class PatchMerging(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(4*dim)
        self.red  = nn.Linear(4*dim, 2*dim, bias=False)
    def forward(self, x, H, W):
        B = x.shape[0]
        x = x.view(B, H, W, -1)
        x = torch.cat([x[:,0::2,0::2,:], x[:,1::2,0::2,:],
                        x[:,0::2,1::2,:], x[:,1::2,1::2,:]], -1)
        x = self.red(self.norm(x.view(B,-1,x.shape[-1])))
        return x, H//2, W//2

# ── FocusFormer Encoder ───────────────────────────────────────────────────
class FocusFormerEncoder(nn.Module):
    def __init__(self, in_c=3, dim=96, ws=8,
                 depths=(2,2,6,2), heads=(3,6,12,24), drop=0.):
        super().__init__()
        self.dim = dim
        self.patch = PatchEmbed(in_c, dim)
        d = [dim, dim*2, dim*4, dim*8]
        self.s1 = nn.ModuleList([FocusFormerBlock(d[0],heads[0],ws,drop=drop) for _ in range(depths[0])])
        self.m1 = PatchMerging(d[0])
        self.s2 = nn.ModuleList([FocusFormerBlock(d[1],heads[1],ws,drop=drop) for _ in range(depths[1])])
        self.m2 = PatchMerging(d[1])
        self.s3 = nn.ModuleList([FocusFormerBlock(d[2],heads[2],ws,drop=drop) for _ in range(depths[2])])
        self.m3 = PatchMerging(d[2])
        self.s4 = nn.ModuleList([FocusFormerBlock(d[3],heads[3],ws,drop=drop) for _ in range(depths[3])])
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight)

    def _sp(self, x, H, W, C):
        return x.view(-1,H,W,C).permute(0,3,1,2).contiguous()

    def forward(self, x):
        d = self.dim
        x, H, W = self.patch(x)
        for b in self.s1: x = b(x, H, W)
        c1 = self._sp(x, H, W, d)
        x, H, W = self.m1(x, H, W)
        for b in self.s2: x = b(x, H, W)
        c2 = self._sp(x, H, W, d*2)
        x, H, W = self.m2(x, H, W)
        for b in self.s3: x = b(x, H, W)
        c3 = self._sp(x, H, W, d*4)
        x, H, W = self.m3(x, H, W)
        for b in self.s4: x = b(x, H, W)
        c4 = self._sp(x, H, W, d*8)
        return [c1, c2, c3, c4]

# ── FPN Decoder ───────────────────────────────────────────────────────────
class FPNDecoder(nn.Module):
    def __init__(self, in_ch=(96,192,384,768), fpn=128):
        super().__init__()
        def lat(c): return nn.Conv2d(c, fpn, 1)
        def sm():   return nn.Sequential(
            nn.Conv2d(fpn,fpn,3,padding=1), nn.BatchNorm2d(fpn), nn.GELU())
        self.l4,self.l3,self.l2,self.l1 = lat(in_ch[3]),lat(in_ch[2]),lat(in_ch[1]),lat(in_ch[0])
        self.s4,self.s3,self.s2,self.s1 = sm(),sm(),sm(),sm()
        self.head = nn.Sequential(
            nn.Conv2d(fpn,64,3,padding=1), nn.BatchNorm2d(64), nn.GELU(),
            nn.Dropout2d(0.1), nn.Conv2d(64,1,1))
    def _up(self, x, ref):
        return F.interpolate(x, size=ref.shape[-2:], mode='bilinear', align_corners=False) + ref
    def forward(self, feats):
        c1,c2,c3,c4 = feats
        p4 = self.s4(self.l4(c4))
        p3 = self.s3(self._up(p4, self.l3(c3)))
        p2 = self.s2(self._up(p3, self.l2(c2)))
        p1 = self.s1(self._up(p2, self.l1(c1)))
        out = F.interpolate(p1, scale_factor=4, mode='bilinear', align_corners=False)
        return self.head(out)

# ── RoadFocusNet ──────────────────────────────────────────────────────────
class RoadFocusNet(nn.Module):
    def __init__(self, dim=96, ws=8, depths=(2,2,6,2),
                 heads=(3,6,12,24), fpn=128, drop=0.):
        super().__init__()
        self.encoder = FocusFormerEncoder(dim=dim, ws=ws, depths=depths, heads=heads, drop=drop)
        self.decoder = FPNDecoder(in_ch=(dim,dim*2,dim*4,dim*8), fpn=fpn)
    def forward(self, x):
        return self.decoder(self.encoder(x))

# ── Sanity check ──────────────────────────────────────────────────────────
print('[TEST] Building RoadFocusNet...')
_m = RoadFocusNet()
_x = torch.randn(2, 3, 512, 512)
_y = _m(_x)
assert _y.shape == (2,1,512,512), f'BAD SHAPE: {_y.shape}'
print(f'[OK]   {tuple(_x.shape)} → {tuple(_y.shape)}')
print(f'[OK]   Params: {sum(p.numel() for p in _m.parameters()):,}')
del _m, _x, _y; gc.collect()
print('[PASS] Sanity check passed.')

## 🧩 3. FocusMIM Pre-Training Head


In [ ]:
class FocusMIMPretrainer(nn.Module):
    def __init__(self, encoder, dim=96):
        super().__init__()
        self.encoder = encoder
        c4 = dim * 8  # 768
        # PixelShuffle(4): 3*16 ch → 3ch at 4× spatial resolution
        self.head = nn.Sequential(
            nn.Conv2d(c4, 256, 3, padding=1), nn.BatchNorm2d(256), nn.GELU(),
            nn.Conv2d(256, 3*16, 1), nn.PixelShuffle(4))

    def forward(self, x_masked):
        c1,c2,c3,c4 = self.encoder(x_masked)
        recon = self.head(c4)
        return F.interpolate(recon, size=x_masked.shape[2:],
                             mode='bilinear', align_corners=False)

print('[OK] FocusMIMPretrainer defined.')

## 📁 4. Dataset Discovery & Preprocessing
Only images with a matching `*_mask.png` are loaded. 80/10/10 split.


In [ ]:
def find_data():
    for p in ['/kaggle/input/deepglobe-road-extraction-dataset',
               '/kaggle/input/deepglobe-road-extraction',
               '/kaggle/input/road-extraction-deepglobe',
               './dataset-Road-Deepglobe', '../dataset-Road-Deepglobe']:
        if os.path.exists(p):
            sats = glob.glob(p+'/*_sat.jpg') + glob.glob(p+'/**/*_sat.jpg', recursive=True)
            if sats: print(f'[DATA] Found {p} — {len(sats)} images'); return p
    raise FileNotFoundError('DeepGlobe dataset not found')

def load_pairs(d):
    sats = sorted(glob.glob(d+'/*_sat.jpg') + glob.glob(d+'/**/*_sat.jpg', recursive=True))
    pairs = [(s, s.replace('_sat.jpg','_mask.png')) for s in sats
              if os.path.exists(s.replace('_sat.jpg','_mask.png'))]
    print(f'[DATA] Matched {len(pairs)} image-mask pairs.'); return pairs

def clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l,a,b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=1.2, tileGridSize=(8,8)).apply(l)
    return cv2.cvtColor(cv2.merge((l,a,b)), cv2.COLOR_LAB2RGB)

def enhance(img):
    img = clahe(img)
    img = cv2.bilateralFilter(img, 5, 25, 25)
    k   = np.array([[0,-.2,0],[-.2,1.8,-.2],[0,-.2,0]], np.float32)
    return np.clip(cv2.filter2D(img,-1,k), 0, 255).astype(np.uint8)

def binarize(mask_rgb):
    return ((mask_rgb[...,0]>128)|(mask_rgb[...,1]>128)|(mask_rgb[...,2]>128)).astype(np.float32)

DATA_DIR  = find_data()
all_pairs = load_pairs(DATA_DIR)
IMG_SIZE  = 512

tr, tmp = train_test_split(all_pairs, test_size=.20, random_state=SEED)
va, te  = train_test_split(tmp,       test_size=.50, random_state=SEED)
print(f'[DATA] Train={len(tr)} | Val={len(va)} | Test={len(te)}')

## 🧪 5. Augmentation, Dataset & DataLoaders


In [ ]:
class FocusMIMAug:
    """Road-aware masked patch augmentation."""
    def __init__(self, p=.5, ps=16, ks=85, ratio=.30, token=128):
        self.p, self.ps, self.ks, self.ratio, self.tok = p, ps, ks, ratio, token

    def __call__(self, img, mask):
        if np.random.rand() > self.p:
            return img.copy(), mask, np.zeros_like(mask)
        H, W = mask.shape
        nph, npw = H//self.ps, W//self.ps
        kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(self.ks,self.ks))
        dil  = cv2.dilate((mask*255).astype(np.uint8), kern)
        road = [(i,j) for i in range(nph) for j in range(npw)
                if dil[i*self.ps:(i+1)*self.ps, j*self.ps:(j+1)*self.ps].max()>0]
        if not road: return img.copy(), mask, np.zeros_like(mask)
        k = max(1, int(len(road)*self.ratio))
        sel = [road[i] for i in np.random.choice(len(road), k, replace=False)]
        pm_s = np.zeros((nph,npw), np.float32)
        for r,c in sel: pm_s[r,c] = 1.
        pm = cv2.resize(pm_s,(W,H), interpolation=cv2.INTER_NEAREST)
        masked = np.where(pm[:,:,None]==1,
                          np.full_like(img,self.tok,np.uint8), img)
        return masked, mask, pm


class RoadDS(Dataset):
    """
    tfm       : standard single-image transform (used for segmentation)
    tfm_mim   : synchronized multi-image transform (used for MIM pre-training)
                Must be created with additional_targets={'clean':'image','patch_mask':'mask'}
    """
    def __init__(self, pairs, tfm=None, tfm_mim=None, train=False, mim=False):
        self.pairs    = pairs
        self.tfm      = tfm
        self.tfm_mim  = tfm_mim
        self.train    = train
        self.mim      = mim
        p = 1. if mim else (.5 if train else 0.)
        self.aug = FocusMIMAug(p=p)

    def __len__(self): return len(self.pairs)

    def _rd(self, p):
        x = cv2.imread(p)
        if x is None: raise FileNotFoundError(p)
        return x

    def __getitem__(self, idx):
        sp, mp = self.pairs[idx]
        img = cv2.resize(cv2.cvtColor(self._rd(sp), cv2.COLOR_BGR2RGB),
                         (IMG_SIZE,IMG_SIZE), interpolation=cv2.INTER_CUBIC)
        img = enhance(img)
        msk = binarize(cv2.resize(
              cv2.cvtColor(self._rd(mp), cv2.COLOR_BGR2RGB),
              (IMG_SIZE,IMG_SIZE), interpolation=cv2.INTER_NEAREST))

        masked, msk, pm = self.aug(img, msk)

        if self.mim:
            # ── KEY FIX: all three tensors share one random transform seed ──
            if self.tfm_mim:
                res  = self.tfm_mim(image=masked, clean=img, patch_mask=pm)
                m_t  = res['image']                                      # masked input
                c_t  = res['clean']                                      # clean target
                pm_t = (res['patch_mask'] > 0.5).float().unsqueeze(0)   # patch mask
            else:
                m_t  = torch.from_numpy(masked.transpose(2,0,1)).float()/255.
                c_t  = torch.from_numpy(img.transpose(2,0,1)).float()/255.
                pm_t = torch.from_numpy(pm).unsqueeze(0)
            return m_t, c_t, pm_t

        src = masked if self.train else img
        if self.tfm:
            a = self.tfm(image=src, mask=msk)
            return a['image'], a['mask'].unsqueeze(0)
        return (torch.from_numpy(src.transpose(2,0,1)).float()/255.,
                torch.from_numpy(msk).unsqueeze(0))


MN, ST = (0.485,.456,.406),(0.229,.224,.225)

# Standard transform for segmentation fine-tuning
tr_tfm = A.Compose([
    A.Resize(IMG_SIZE,IMG_SIZE),
    A.HorizontalFlip(p=.5), A.VerticalFlip(p=.5), A.RandomRotate90(p=.5),
    A.ShiftScaleRotate(.05,.1,15,p=.4),
    A.OneOf([A.RandomBrightnessContrast(.15,.15),A.HueSaturationValue(10,20,10)],p=.4),
    A.GridDistortion(p=.3), A.Normalize(MN,ST), ToTensorV2()])

# Synchronized MIM transform: masked img + clean img + patch mask → same spatial ops
# additional_targets ensures all three inputs share a single random seed
tr_tfm_mim = A.Compose([
    A.Resize(IMG_SIZE,IMG_SIZE),
    A.HorizontalFlip(p=.5), A.VerticalFlip(p=.5), A.RandomRotate90(p=.5),
    A.ShiftScaleRotate(.05,.1,15,p=.4),
    A.GridDistortion(p=.3), A.Normalize(MN,ST), ToTensorV2(),
], additional_targets={'clean': 'image', 'patch_mask': 'mask'})

va_tfm = A.Compose([A.Resize(IMG_SIZE,IMG_SIZE), A.Normalize(MN,ST), ToTensorV2()])

BS = 4 if device.type=='cuda' else 2
NW = 2 if device.type=='cuda' else 0
PW = NW > 0

def mk_loader(ds, shuffle, drop=False):
    return DataLoader(ds, BS, shuffle=shuffle, num_workers=NW,
                      pin_memory=True, drop_last=drop, persistent_workers=PW)

tr_ds  = RoadDS(tr, tfm=tr_tfm, train=True)
va_ds  = RoadDS(va, tfm=va_tfm)
te_ds  = RoadDS(te, tfm=va_tfm)
tr_ld  = mk_loader(tr_ds, True,  drop=True)
va_ld  = mk_loader(va_ds, False)
te_ld  = mk_loader(te_ds, False)
print(f'[OK] Loaders: Train={len(tr_ld)} | Val={len(va_ld)} | Test={len(te_ld)} batches')

## 📐 6. Loss Functions & Metrics


In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, s=1.): super().__init__(); self.s=s
    def forward(self, lg, t):
        p=torch.sigmoid(lg).view(-1); t=t.view(-1)
        return 1.-(2.*(p*t).sum()+self.s)/(p.sum()+t.sum()+self.s)

class SegLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce=nn.BCEWithLogitsLoss(); self.dice=DiceLoss()
    def forward(self,lg,t): return .5*self.bce(lg,t)+.5*self.dice(lg,t)

def mim_loss(recon, tgt, mask):
    """MSE only on masked pixels (not averaged over full image)."""
    m3 = mask.expand_as(recon)
    return ((recon-tgt)**2 * m3).sum() / m3.sum().clamp(min=1)

@torch.no_grad()
def metrics(prob, tgt, thr=.5, eps=1e-6):
    p=(prob>thr).float().view(-1); t=tgt.view(-1)
    tp=(p*t).sum().item(); fp=(p*(1-t)).sum().item(); fn=((1-p)*t).sum().item()
    return (tp+eps)/(tp+fp+fn+eps), (2*tp+eps)/(2*tp+fp+fn+eps)

print('[OK] Loss & metrics ready.')

## ⚡ 7. Phase 1 — FocusMIM Self-Supervised Pre-Training (10 Epochs)
Includes a 2-epoch linear learning rate warmup for smooth transformer convergence.


In [ ]:
mim_ds = RoadDS(tr, tfm_mim=tr_tfm_mim, train=True, mim=True)
mim_ld = mk_loader(mim_ds, True, drop=True)

model      = RoadFocusNet(dim=96, ws=8, depths=(2,2,6,2),
                           heads=(3,6,12,24), fpn=128, drop=.1).to(device)
pretrainer = FocusMIMPretrainer(model.encoder, dim=96).to(device)

MIM_LR     = 5e-4
MIM_EP     = 10
MIM_WU     = 2  # 2-epoch warmup for smooth transformer learning
mim_opt    = torch.optim.AdamW(pretrainer.parameters(), lr=MIM_LR, weight_decay=1e-4)
mim_sched  = torch.optim.lr_scheduler.CosineAnnealingLR(mim_opt, T_max=MIM_EP-MIM_WU, eta_min=1e-5)
mim_scaler = make_scaler()
MIM_CKPT   = 'focusformer_encoder.pth'
mim_log    = []

print('='*65)
print('PHASE 1 — FocusMIM Pre-Training (Warmup=2ep, Cosine Decay=8ep)')
print('='*65)
t0 = time.time()

for ep in range(1, MIM_EP+1):
    # Linear warmup for first 2 epochs
    if ep <= MIM_WU:
        for g in mim_opt.param_groups: g['lr'] = MIM_LR * ep / MIM_WU

    pretrainer.train(); run=0.; n_b=0
    bar = tqdm(mim_ld, desc=f'MIM {ep:02d}/{MIM_EP}', leave=False)
    for msk_t, cln_t, pm_t in bar:
        msk_t = msk_t.to(device, non_blocking=True)
        cln_t = cln_t.to(device, non_blocking=True)
        pm_t  = pm_t.to(device,  non_blocking=True)
        if pm_t.sum() < 1: continue
        mim_opt.zero_grad()
        with autocast():
            loss = mim_loss(pretrainer(msk_t), cln_t, pm_t)
        if torch.isnan(loss) or torch.isinf(loss): continue
        mim_scaler.scale(loss).backward()
        mim_scaler.unscale_(mim_opt)
        nn.utils.clip_grad_norm_(pretrainer.parameters(), 1.0)
        mim_scaler.step(mim_opt); mim_scaler.update()
        run += loss.item(); n_b += 1
        bar.set_postfix(loss=f'{loss.item():.5f}')
    if ep > MIM_WU:
        mim_sched.step()
    avg = run/max(1,n_b); mim_log.append(avg)
    print(f'  Ep{ep:02d}  loss={avg:.5f}  lr={mim_opt.param_groups[0]["lr"]:.2e}')

torch.save(model.encoder.state_dict(), MIM_CKPT)
print(f'[SAVED] {MIM_CKPT}  ({(time.time()-t0)/60:.1f} min)')

## 🎯 8. Phase 2 — Supervised Fine-Tuning (50 Epochs)


In [ ]:
def find_ckpt(name='best_v4.pth'):
    if os.path.exists(name): return name
    for p in glob.glob(f'/kaggle/input/**/{name}', recursive=True):
        if os.path.exists(p): print(f'[CKPT] Found: {p}'); return p
    return name

FT_EP  = 50
WU_EP  = 3
LR     = 1e-4
CKPT   = 'best_v4.pth'

crit      = SegLoss().to(device)
opt       = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
sched     = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FT_EP-WU_EP, eta_min=1e-6)
scaler    = make_scaler()
best_iou  = 0.; hist={'tl':[],'vl':[],'vi':[],'vd':[]}

print('='*65)
print(f'PHASE 2 — Fine-Tuning  EP={FT_EP} BS={BS} AMP={USE_AMP}')
print('='*65)
t0=time.time()

for ep in range(1, FT_EP+1):
    if ep <= WU_EP:
        for g in opt.param_groups: g['lr'] = LR*ep/WU_EP

    model.train(); tl=0.
    for imgs,msks in tqdm(tr_ld, desc=f'Tr {ep:02d}/{FT_EP}', leave=False):
        imgs=imgs.to(device,non_blocking=True)
        msks=msks.to(device,non_blocking=True)
        opt.zero_grad()
        with autocast(): loss=crit(model(imgs), msks)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.)
        scaler.step(opt); scaler.update()
        tl+=loss.item()
    if ep>WU_EP: sched.step()

    model.eval(); vl=vi=vd=0.
    with torch.no_grad():
        for imgs,msks in va_ld:
            imgs=imgs.to(device,non_blocking=True)
            msks=msks.to(device,non_blocking=True)
            with autocast(): lg=model(imgs); vl+=crit(lg,msks).item()
            i,d=metrics(torch.sigmoid(lg),msks); vi+=i; vd+=d

    n=len(tr_ld); m=len(va_ld)
    tl/=n; vl/=m; vi/=m; vd/=m
    hist['tl'].append(tl); hist['vl'].append(vl)
    hist['vi'].append(vi); hist['vd'].append(vd)

    tag=''
    if vi>best_iou:
        best_iou=vi; torch.save(model.state_dict(), CKPT); tag=' ⭐'
    lr_now=opt.param_groups[0]['lr']
    print(f'Ep{ep:02d}  tl={tl:.4f} vl={vl:.4f} IoU={vi:.4f} Dice={vd:.4f} lr={lr_now:.2e}{tag}')

print('='*65)
print(f'Best Val IoU: {best_iou:.4f}  ({(time.time()-t0)/60:.1f} min)')

## 📊 9. Convergence Plots


In [ ]:
fig,(a0,a1,a2)=plt.subplots(1,3,figsize=(18,5))
fig.suptitle('V4.0 RoadFocusNet — Convergence',fontsize=14,fontweight='bold')
a0.plot(mim_log,'o-',c='#e74c3c',lw=2)
a0.set(title='Phase 1 MIM Loss',xlabel='Epoch',ylabel='MSE (masked only)'); a0.grid(.3)
xs=range(1,FT_EP+1)
a1.plot(xs,hist['tl'],label='Train',c='#3498db',lw=2)
a1.plot(xs,hist['vl'],label='Val',  c='#e74c3c',lw=2)
a1.set(title='Phase 2 Loss',xlabel='Epoch'); a1.legend(); a1.grid(.3)
a2.plot(xs,hist['vi'],label='IoU', c='#2ecc71',lw=2)
a2.plot(xs,hist['vd'],label='Dice',c='#9b59b6',lw=2)
a2.axhline(.6637,ls='--',c='#e74c3c',lw=1,label='V1 baseline')
a2.set(title='Phase 2 Metrics',xlabel='Epoch'); a2.legend(fontsize=8); a2.grid(.3)
plt.tight_layout()
plt.savefig('v4_convergence.png',dpi=150,bbox_inches='tight'); plt.show()

## 🔬 10. Visual Predictions (10 Test Samples)


In [ ]:
model.load_state_dict(torch.load(find_ckpt(CKPT), map_location=device, weights_only=False))
model.eval()
n_show = min(10, len(te))
fig,axes=plt.subplots(n_show,4,figsize=(20,4.5*n_show))
fig.suptitle('V4.0 FocusFormer — Test Predictions',fontsize=16,fontweight='bold')

for i,(sp,mp) in enumerate(te[:n_show]):
    img=cv2.resize(cv2.cvtColor(cv2.imread(sp),cv2.COLOR_BGR2RGB),(IMG_SIZE,IMG_SIZE))
    prep=enhance(img)
    gt=binarize(cv2.resize(cv2.cvtColor(cv2.imread(mp),cv2.COLOR_BGR2RGB),
                            (IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST))
    aug=va_tfm(image=prep,mask=gt)
    with torch.no_grad():
        with autocast():
            prob=torch.sigmoid(model(aug['image'].unsqueeze(0).to(device)))
    pred=(prob.squeeze().cpu().float().numpy()>.5).astype(np.float32)
    iou=(np.logical_and(pred,gt).sum()+1e-6)/(np.logical_or(pred,gt).sum()+1e-6)
    ov=img.copy(); ov[pred==1]=[0,230,100]
    bl=cv2.addWeighted(img,.65,ov,.35,0)
    axes[i,0].imshow(img);            axes[i,0].set_title(f'#{i+1} Input');    axes[i,0].axis('off')
    axes[i,1].imshow(gt,cmap='gray'); axes[i,1].set_title('Ground Truth');     axes[i,1].axis('off')
    axes[i,2].imshow(pred,cmap='gray');axes[i,2].set_title('Prediction');      axes[i,2].axis('off')
    axes[i,3].imshow(bl);             axes[i,3].set_title(f'IoU={iou:.4f}');   axes[i,3].axis('off')
plt.tight_layout()
plt.savefig('v4_preds.png',dpi=120,bbox_inches='tight'); plt.show()

In [ ]:
model.load_state_dict(torch.load(find_ckpt(CKPT), map_location=device, weights_only=False))
model.eval()
ious,dices=[],[]
with torch.no_grad():
    for imgs,msks in tqdm(te_ld,'Test'):
        with autocast():
            p=torch.sigmoid(model(imgs.to(device,non_blocking=True)))
        i,d=metrics(p, msks.to(device,non_blocking=True))
        ious.append(i); dices.append(d)
ti,td=float(np.mean(ious)),float(np.mean(dices))
print(f'\nTest IoU={ti:.4f}  Dice={td:.4f}')

lb=pd.DataFrame({'Model':['V1 DeepLabV3+','V4 FocusFormer'],
                  'Backbone':['ResNet-34','FocusFormer (W-FSA+CSA)'],
                  'Val IoU':[.6637,round(best_iou,4)],
                  'Test IoU':[.6637,round(ti,4)],
                  'Test Dice':[.7970,round(td,4)]})
print('\n🏆 Leaderboard:')
print(lb.to_string(index=False))

## 📋 11. Final Test-Set Evaluation


In [ ]:
model.load_state_dict(torch.load(CKPT, map_location=device, weights_only=False))
model.eval()
ious,dices=[],[]
with torch.no_grad():
    for imgs,msks in tqdm(te_ld,'Test'):
        with autocast():
            p=torch.sigmoid(model(imgs.to(device,non_blocking=True)))
        i,d=metrics(p, msks.to(device,non_blocking=True))
        ious.append(i); dices.append(d)
ti,td=float(np.mean(ious)),float(np.mean(dices))
print(f'\nTest IoU={ti:.4f}  Dice={td:.4f}')

lb=pd.DataFrame({'Model':['V1 DeepLabV3+','V4 FocusFormer'],
                  'Backbone':['ResNet-34','FocusFormer (W-FSA+CSA)'],
                  'Val IoU':[.6637,round(best_iou,4)],
                  'Test IoU':[.6637,round(ti,4)],
                  'Test Dice':[.7970,round(td,4)]})
print('\n🏆 Leaderboard:')
print(lb.to_string(index=False))

## 💾 12. Download Checkpoint


In [ ]:
from IPython.display import FileLink, display
if os.path.exists(CKPT):
    print(f'{CKPT}: {os.path.getsize(CKPT)/1024/1024:.1f} MB')
    display(FileLink(CKPT))
    print('\n🎉 V4.0 complete!')